# Transición Dogs of the World · `pettrace-training@v4.100` → `dogs-of-the-world-refined@v1`

**Objetivo.** Publicar una copia portable y verificable de la porción Dogs of the World refinada que ya usaron los experimentos.

**Entrada inmutable.** `pettrace-training@v4.100`, fingerprint `d0ca5c0b1bed8844f943b9fd355ce912524797d0dc71b50c7a1624e39f49fe58`.

**Salida.** Dataset público `alessandro54/dogs-of-the-world-refined@v1`, bajo `cc0-1.0`, la licencia declarada por el publicador.

Este cuaderno no vuelve a limpiar imágenes, no ejecuta YOLO, no entrena y no publica resultados MLflow.

## Fase 1. Recuperar y verificar la entrada

Ejecuta en el mismo runtime de Colab. Drive conserva `v4.100` y el staging reanudable; `HF_TOKEN` se lee desde Colab Secrets.

In [ ]:
import subprocess, sys
from importlib.metadata import version as package_version

pillow_version = package_version("Pillow")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
                       "pettrace-ml==0.2.16", f"Pillow=={pillow_version}"])

In [ ]:
import json
import os
from pathlib import Path

from google.colab import drive, userdata
from pettrace import data

EXPECTED_FINGERPRINT = "d0ca5c0b1bed8844f943b9fd355ce912524797d0dc71b50c7a1624e39f49fe58"
CACHE = Path("/content/datasets/build/pettrace-training")
DRIVE_ROOT = Path("/content/drive/MyDrive/PetTrace")
STAGING = DRIVE_ROOT / "releases/dogs-of-the-world-refined-v1"
REFERENCE = CACHE / "v4.100.json"

def remount_drive():
    try:
        drive.flush_and_unmount()
    except OSError:
        pass
    drive.mount("/content/drive", force_remount=True)

try:
    drive_ready = DRIVE_ROOT.parent.is_dir()
except OSError:
    remount_drive()
else:
    if not drive_ready:
        drive.mount("/content/drive")
token = userdata.get("HF_TOKEN")
if not token:
    raise RuntimeError("Falta HF_TOKEN en Colab Secrets o el notebook no tiene acceso.")
os.environ["HF_TOKEN"] = token

try:
    prepared, provenance = data.load("v4.100", cache_dir=CACHE, drive_root=DRIVE_ROOT)
except OSError as exc:
    if exc.errno not in {103, 107}:
        raise
    print("Drive perdió su conexión; remontando y reanudando una vez…")
    remount_drive()
    prepared, provenance = data.load("v4.100", cache_dir=CACHE, drive_root=DRIVE_ROOT)
if prepared.fingerprint != EXPECTED_FINGERPRINT:
    raise ValueError("v4.100 no coincide con el fingerprint congelado.")
if not REFERENCE.is_file():
    raise FileNotFoundError("No se creó la referencia inmutable v4.100.json.")
print(f"Entrada verificada: {prepared.fingerprint} | fotos: {len(prepared.rows):,}")
print(f"Procedencia: {provenance.get('repo')}@{provenance.get('version')}")

## Fase 2. Construir y auditar `dogs-of-the-world-refined@v1`

La herramienta selecciona únicamente `dogs-world`, crea shards deterministas por split y escribe hashes, metadatos, licencia y procedencia. Todavía no modifica Hugging Face.

In [ ]:
import hashlib
import subprocess
import sys
from urllib.request import urlretrieve

TOOL_URL = "https://raw.githubusercontent.com/alessandro54/pettrace-ml/main/tools/publish_hf_dataset.py"
TOOL_SHA256 = "64ee694b467d6acb3e4528ed7e70d9ff978b984350dda85632ed6b968acd6990"
TOOL = Path("/content/publish_hf_dataset.py")
urlretrieve(TOOL_URL, TOOL)
if hashlib.sha256(TOOL.read_bytes()).hexdigest() != TOOL_SHA256:
    raise ValueError("La herramienta de publicación no coincide con la revisión auditada.")
subprocess.run([sys.executable, str(TOOL), "--prepared", str(REFERENCE),
                "--staging", str(STAGING), "--source", "dogs-world"], check=True)

In [ ]:
import csv
from IPython.display import Markdown, display

evidence = json.loads((STAGING / "versions/v1/dataset.json").read_text())
with (STAGING / "versions/v1/metadata.csv").open() as stream:
    preview = list(csv.DictReader(stream))[:5]
display(Markdown(
    f"**Release preparada:** `{evidence['source']}@{evidence['release_version']}`  \\n"
    f"**Origen:** `pettrace-training@{evidence['prepared_version']}`  \\n"
    f"**Fotos:** {evidence['photos']:,} · **Identidades:** {evidence['identities']:,}  \\n"
    f"**Splits:** `{evidence['splits']}` · **Shards:** {len(evidence['shards'])}  \\n"
    f"**Licencia:** `{evidence['license']}`"
))
display(preview)

## Fase 3. Publicar y verificar la release

Esta es la única mutación externa: publica el staging auditado, verifica el inventario remoto y crea el tag inmutable `v1`.

In [ ]:
from huggingface_hub import HfApi

REPO_ID = "alessandro54/dogs-of-the-world-refined"
subprocess.run([sys.executable, str(TOOL), "--prepared", str(REFERENCE),
                "--staging", str(STAGING), "--source", "dogs-world",
                "--repo-id", REPO_ID, "--upload"], check=True)
api = HfApi(token=os.environ["HF_TOKEN"])
remote = api.dataset_info(REPO_ID, revision="v1")
remote_files = api.list_repo_files(REPO_ID, repo_type="dataset", revision=remote.sha)
required = {"README.md", "LICENSE", "versions/v1/dataset.json", "versions/v1/metadata.csv"}
if not required <= set(remote_files):
    raise RuntimeError("La release remota quedó incompleta.")
print(f"Publicada y verificada: https://huggingface.co/datasets/{REPO_ID}/tree/v1")
print(f"Commit: {remote.sha} | archivos: {len(remote_files):,}")